# Эксперименты с гиперпараметрами — YOLOv8 и SSD

Эксперименты проводим на **2 моделях**: **YOLOv8** и **SSD** — самые дешёвые по времени. Остальные 3 (Faster R-CNN, EfficientDet, DETR) остаются на дефолтных гиперпараметрах.

Все конфиги: **train 3000, 20 эпох, оценка на полном test** — прямо сравнимы с baseline и между собой. Варьируем по одному ключевому параметру:
- **YOLO:** оптимизатор (Adam/SGD), learning rate, аугментация (mosaic);
- **SSD:** learning rate, оптимизатор (SGD/Adam).

Итог — таблицы «конфиг → метрики» для §3.5 (выбор финальных гиперпараметров) и §3.6. 

## 0. Окружение

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
import torch, torchvision
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Репозиторий

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/artemdev2281/cv-fashion-object-detection.git"  
REPO_DIR = "/kaggle/working/cv-fashion-object-detection"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0]:", sys.path[0])

## 2. Данные, классы и подвыборка 3000

In [ ]:
import json, yaml
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/notebooks/neuromindgpt/"
    "cv-fashion-object-detection/cv-fashion-object-detection/data/processed"
)
assert DATA_ROOT.exists(), f"Не найден каталог данных: {DATA_ROOT}. Проверьте Add Input."

ANN = {s: DATA_ROOT / "coco" / f"{s}.json" for s in ("train", "val", "test")}
classes = json.load(open(DATA_ROOT / "classes.json", encoding="utf-8"))
CLASS_NAMES = [classes["names"][str(i)] for i in range(classes["num_classes"])]
NUM_CLASSES_NO_BG = classes["num_classes"]      # 11
NUM_CLASSES = NUM_CLASSES_NO_BG + 1             # 12 (с фоном) — для SSD/torchvision

# Подвыборка 3000 для YOLO (те же файлы, что у torchvision): первые 3000 image_id.
data_yaml_full = yaml.safe_load(open(DATA_ROOT / "data.yaml", encoding="utf-8"))
data_yaml_full["path"] = str(DATA_ROOT)
train_coco = json.load(open(ANN["train"], encoding="utf-8"))
ids = sorted(img["id"] for img in train_coco["images"])[:3000]
id_to_file = {img["id"]: img["file_name"] for img in train_coco["images"]}
subset_txt = Path("/kaggle/working/train_subset_3000.txt")
with open(subset_txt, "w", encoding="utf-8") as f:
    for i in ids:
        f.write(str(DATA_ROOT / id_to_file[i]) + "\n")
data_sub = dict(data_yaml_full); data_sub["train"] = str(subset_txt)
SUBSET_YAML = Path("/kaggle/working/data_subset3000.yaml")
yaml.safe_dump(data_sub, open(SUBSET_YAML, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

HP_DIR = "/kaggle/working/results/hp"
print("Классов:", NUM_CLASSES_NO_BG, "| подвыборка train:", len(ids))

## 3. Конфигурация

In [ ]:
from src.utils.utils import load_config
config = load_config(Path(REPO_DIR) / "configs" / "default.yaml")
print("seed:", config["training"]["seed"])

## 4. YOLOv8 — сетка гиперпараметров

Варьируем оптимизатор, learning rate и аугментацию mosaic. Первый конфиг — дефолтный (совпадает с baseline).

In [ ]:
from src.models.yolo import build_model as build_yolo
from src.training.train import train

yolo_configs = [
    ("yolo_adam_lr001",  dict(optimizer="Adam", lr0=0.001)),               # baseline (дефолт)
    ("yolo_sgd_lr01",    dict(optimizer="SGD",  lr0=0.01)),                # другой оптимизатор + lr
    ("yolo_adam_lr005",  dict(optimizer="Adam", lr0=0.005)),              # выше lr
    ("yolo_adam_mosaic", dict(optimizer="Adam", lr0=0.001, mosaic=1.0)),  # + mosaic-аугментация
]

results_yolo = {}
for name, overrides in yolo_configs:
    print(f"\n=== {name}: {overrides} ===")
    model = build_yolo(NUM_CLASSES_NO_BG, config=config, data_yaml=str(SUBSET_YAML))
    m = train(model, str(SUBSET_YAML), config, epochs=20, split="test",
              project=HP_DIR, name=name, **overrides)
    results_yolo[name] = m
    print(f"{name}: mAP@0.5={m['map50']:.4f}  mAP@0.5:0.95={m['map50_95']:.4f}  F1={m['f1']:.4f}")

In [ ]:
import pandas as pd

def summary_table(results):
    rows = {name: {k: m[k] for k in ("map50", "map50_95", "precision", "recall", "f1")}
            for name, m in results.items()}
    return pd.DataFrame(rows).T.sort_values("map50", ascending=False).round(4)

tbl_yolo = summary_table(results_yolo)
print("YOLOv8 — влияние гиперпараметров (test):")
tbl_yolo

## 5. SSD — сетка гиперпараметров

Варьируем learning rate и оптимизатор. У Adam lr на порядок меньше, чем у SGD.

In [ ]:
from src.models.ssd import build_model as build_ssd
from src.training.train import run_torchvision_training

ssd_configs = [
    ("ssd_sgd_lr005",   dict(optimizer_name="sgd",  lr=0.005, batch_size=8)),   # baseline
    ("ssd_sgd_lr001",   dict(optimizer_name="sgd",  lr=0.001, batch_size=8)),   # ниже lr
    ("ssd_adam_lr0005", dict(optimizer_name="adam", lr=0.0005, batch_size=8)),  # Adam
]

results_ssd = {}
for name, kw in ssd_configs:
    print(f"\n=== {name}: {kw} ===")
    m = run_torchvision_training(
        build_ssd, NUM_CLASSES, ANN["train"], ANN["test"], DATA_ROOT, config,
        model_name=name, class_names=CLASS_NAMES,
        epochs=20, subset_size=3000, project=HP_DIR, **kw,
    )
    results_ssd[name] = m
    print(f"{name}: mAP@0.5={m['map50']:.4f}  mAP@0.5:0.95={m['map50_95']:.4f}  F1={m['f1']:.4f}")

In [ ]:
tbl_ssd = summary_table(results_ssd)
print("SSD — влияние гиперпараметров (test):")
tbl_ssd

## 6. Итоги и сохранение

Лучшие конфиги пойдут в §3.5 (финальные гиперпараметры для YOLO и SSD). 

In [ ]:
import json, shutil
from pathlib import Path

best_yolo = tbl_yolo.index[0]
best_ssd = tbl_ssd.index[0]
print(f"Лучший YOLO: {best_yolo}  (mAP@0.5={tbl_yolo.loc[best_yolo, 'map50']})")
print(f"Лучший SSD:  {best_ssd}  (mAP@0.5={tbl_ssd.loc[best_ssd, 'map50']})")

out = Path("/kaggle/working/results")
out.mkdir(parents=True, exist_ok=True)
json.dump({"yolo": results_yolo, "ssd": results_ssd},
          open(out / "hp_experiments.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
tbl_yolo.to_csv(out / "hp_yolo.csv", encoding="utf-8")
tbl_ssd.to_csv(out / "hp_ssd.csv", encoding="utf-8")
shutil.make_archive("/kaggle/working/hp_experiments_results", "zip", "/kaggle/working/results")
print("Сохранено. Скачайте /kaggle/working/hp_experiments_results.zip из Output и/или Save Version.")